# 05 · 实际计算 Fisher / CRLB
1D Gaussian 像素积分，参数 $\theta=(x,N,b)$：位置 nm、ROI 前的期望信号 photons/frame、背景 photons/pixel/frame。有限 ROI 不重新归一化，因此边缘附近的漏光进入模型。给定参数下像素 Poisson 独立。

$F_{ij}=\sum_p(\partial_i\lambda_p)(\partial_j\lambda_p)/\lambda_p$。
MC 估计是固定采样后，对 log likelihood 求 score；不是对采样器求导。


In [ ]:
import torch, math
import matplotlib.pyplot as plt
torch.set_num_threads(2); torch.set_default_dtype(torch.float64); torch.manual_seed(7)
edges=torch.linspace(-450,450,10); sigma=120.
def rate(theta):
    x,N,b=theta
    cdf=.5*(1+torch.erf((edges-x)/(sigma*math.sqrt(2))))
    return N*(cdf[1:]-cdf[:-1])+b
theta=torch.tensor([20.,100.,2.])
lam=rate(theta)
J=torch.autograd.functional.jacobian(rate,theta)
F=J.T@(J/lam[:,None])
eigen=torch.linalg.eigvalsh(F)
assert eigen.min()>0
C=torch.linalg.solve(F,torch.eye(3))
print('FIM eigenvalues (parameter units differ):',eigen)
print('known N,b position SD bound (nm):',float(F[0,0].rsqrt()))
print('unknown N,b position SD bound (nm):',float(C[0,0].sqrt()))
print('condition number depends on parameter scaling:',float(torch.linalg.cond(F)))


In [ ]:
with torch.no_grad():
    samples=torch.poisson(lam.expand(4000,-1)).detach()
def loglik(th,obs):
    return torch.distributions.Poisson(rate(th)).log_prob(obs).sum()
from torch.func import grad, vmap
scores=vmap(grad(loglik),in_dims=(None,0))(theta,samples)
Fmc=scores.T@scores/len(samples)
# Independent block estimates give an empirical MC standard error.
blocks=scores.reshape(20,200,3)
Fb=torch.einsum('bni,bnj->bij',blocks,blocks)/200
se=Fb.std(dim=0)/math.sqrt(20)
print('analytic F:',F,'MC F:',Fmc,'MC standard error:',se)
print('mean score (should approach zero):',scores.mean(0))
# Finite difference validates derivatives of the deterministic rate.
for j,h in enumerate([1e-3,1e-3,1e-5]):
    d=torch.zeros(3); d[j]=h
    fd=(rate(theta+d)-rate(theta-d))/(2*h)
    assert torch.allclose(fd,J[:,j],rtol=1e-5,atol=1e-7)


## 必须解释
为什么 $[F^{-1}]_{xx}$ 通常大于 $1/F_{xx}$？N,b 是 nuisance parameters 时用全矩阵逆或 Schur complement。不要对接近奇异的 F 无条件加 ridge 并叫它 CRLB。

增加重复帧若独立且参数共享，则 F 相加；共享 blinking 隐状态导致边缘观测相关，不能直接按像素/帧相加。

CRLB 是满足正则条件的局部无偏估计协方差下界，不是任何 neural estimator 的 RMSE 下界；低 photon 时 MLE 有偏、有限样本效率差，可能达不到界。
